# FANT 3 — Colab training notebook

Parameterized by `TARGET_SCALE` — start at 50M, scale up once the loop is healthy.

**Supported scales:** `50m`, `150m`, `350m`, `742m`, `1b`.

All five research-driven fixes landed 2026-04-19 are active in this notebook:
1. `<answer>`-wrapped training targets (format matches eval)
2. `tokenizer_v2.json` (retrained on the 6-source distillation mix, 10-18% better compression)
3. `SpinorApollonianMemory` (Kocik chirality split — fixes α/β starvation)
4. Artificial Hippocampus Network gate (sliding + compressed long-term residual)
5. SAE diagnostics (attached as optional mid-training probe)

### Before you start
1. **Upload the project** — easiest route: zip `fant2/`, `fant3/`, `bendvm/`, `output/tokenizer/`, `tests/` from your local `D:\FANT_TRAINING_D_Drive\fant2\` and upload as `fant_code.zip` to Google Drive, e.g. at `MyDrive/fant_code.zip`.
2. **Enable a GPU runtime** — Runtime → Change runtime type → A100 (Pro+ / PAYG) is the sweet spot. T4 works but is ~3× slower than an A100.
3. **Run cells top-to-bottom** the first time. Subsequent sessions: start from the Drive-mount cell (saved checkpoints resume automatically).

## 1. GPU + environment check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
# CRITICAL: must be set BEFORE torch imports / CUDA init, otherwise no-op
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import torch
print(f"torch {torch.__version__}  cuda {torch.version.cuda}  bf16 supported {torch.cuda.is_bf16_supported()}")

## 2. Install deps

Most of these are preinstalled on Colab; we just pin versions and add bitsandbytes for 8-bit AdamW.

In [ ]:
!pip install -q --upgrade bitsandbytes>=0.43 tokenizers>=0.15 datasets>=2.18
# sanity
import bitsandbytes as bnb
import tokenizers, datasets
print(f"bitsandbytes {bnb.__version__}  tokenizers {tokenizers.__version__}  datasets {datasets.__version__}")

## 3. Mount Google Drive and locate code

The notebook looks for `fant_code.zip` at `MyDrive/fant_code.zip` by default. Change `CODE_ZIP` if yours is elsewhere.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib, zipfile, shutil

CODE_ZIP      = '/content/drive/MyDrive/fant_code.zip'
WORK_DIR      = '/content/fant'
CKPT_DIR      = '/content/drive/MyDrive/fant_ckpts'
HF_CACHE_DIR  = '/content/drive/MyDrive/fant_hf_cache'

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['HF_DATASETS_CACHE'] = HF_CACHE_DIR

if not os.path.exists(os.path.join(WORK_DIR, 'fant3', 'model', 'fant3_model.py')):
    assert os.path.exists(CODE_ZIP), f'Code zip not found at {CODE_ZIP}. Upload fant_code.zip to Drive.'
    os.makedirs(WORK_DIR, exist_ok=True)
    with zipfile.ZipFile(CODE_ZIP) as z:
        z.extractall(WORK_DIR)
    print(f'Extracted code to {WORK_DIR}')
else:
    print(f'Code already present at {WORK_DIR}')

import sys
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

# Verify imports
from fant3.config       import FANT3Config, fant3_smoke, fant3_20m, fant3_50m, fant3_742m
from fant3.model.fant3_model import FANT3Model
from fant3.model.spinor_apollonian import SpinorApollonianMemory
from fant3.model.ahn    import ArtificialHippocampusNetwork
from fant2.tokenizer.bpe import FANT2Tokenizer
from fant2.data.formats import extract_text, DatasetFormat
print('All imports OK')

## 4. Pick a scale

Start at `'50m'` to verify end-to-end in the Colab environment. Scale up by changing this single constant and re-running from this cell down.

In [ ]:
TARGET_SCALE = '50m'   # one of '20m', '50m', '150m', '350m', '742m', '1b'

def cfg_20m():
    # Uses fant3_20m() preset — ~23.5M stored, chat-focused
    return fant3_20m()

def cfg_50m():
    # Uses fant3_50m() preset — 50.79M stored, chat-optimized (12h A100 96GB)
    return fant3_50m()

def cfg_150m():
    return FANT3Config(
        dim=768, n_layers=10, n_dense_layers=2,
        n_heads=8, n_kv_heads=2, head_dim=96,
        n_megapools=2, n_per_megapool=4, top_k=2,
        n_matryoshka_levels=2,
        shared_expert_hidden=384, moe_hidden=768,
        n_attention_atoms=3, masa_coef_rank=8,
        n_recursion_depths=2,
        kron_A_p=24, kron_A_q=8, kron_B_p=16, kron_B_q=16, kron_C_p=24, kron_C_q=24,
        max_seq_len=1024,
        cerebellum_enabled=False,
        apollonian_alpha_cap=2000, apollonian_beta_cap=2000,
        apollonian_retrieval_layers=(8, 9),
        etf_freeze_after_step=800,
        etf_freeze_layers=tuple(range(2, 8)),
        spinor_apollonian_enabled=True,
        ahn_enabled=True,
        ahn_n_heads=2, ahn_short_window=32, ahn_long_capacity=64,
    )

def cfg_350m():
    return FANT3Config(
        dim=1024, n_layers=14, n_dense_layers=2,
        n_heads=8, n_kv_heads=2, head_dim=128,
        n_megapools=4, n_per_megapool=4, top_k=2,
        n_matryoshka_levels=2,
        shared_expert_hidden=512, moe_hidden=1024,
        n_attention_atoms=4, masa_coef_rank=8,
        n_recursion_depths=2,
        kron_A_p=32, kron_A_q=8, kron_B_p=16, kron_B_q=16, kron_C_p=32, kron_C_q=32,
        max_seq_len=1024,
        cerebellum_enabled=False,
        apollonian_alpha_cap=3000, apollonian_beta_cap=3000,
        apollonian_retrieval_layers=(12, 13),
        etf_freeze_after_step=1000,
        etf_freeze_layers=tuple(range(2, 11)),
        spinor_apollonian_enabled=True,
        ahn_enabled=True,
        ahn_n_heads=4, ahn_short_window=64, ahn_long_capacity=128,
    )

def cfg_742m():
    c = fant3_742m()
    c.spinor_apollonian_enabled = True
    c.ahn_enabled = True
    c.ahn_n_heads = 4; c.ahn_short_window = 128; c.ahn_long_capacity = 256
    c.cerebellum_enabled = False
    return c

def cfg_1b():
    c = FANT3Config()
    c.spinor_apollonian_enabled = True
    c.ahn_enabled = True
    c.ahn_n_heads = 4; c.ahn_short_window = 128; c.ahn_long_capacity = 256
    c.cerebellum_enabled = False
    return c

CONFIG_BUILDERS = {
    '20m': cfg_20m, '50m': cfg_50m, '150m': cfg_150m, '350m': cfg_350m,
    '742m': cfg_742m, '1b': cfg_1b,
}

cfg = CONFIG_BUILDERS[TARGET_SCALE]()
print(f'Target scale: {TARGET_SCALE}')
print(f'dim={cfg.dim} layers={cfg.n_layers} megapools={cfg.n_megapools}x{cfg.n_per_megapool} topk={cfg.top_k}')

## 5. Load tokenizer

Uses `tokenizer_v2.json` (10-18% better compression than the old tokenizer on prose/math/JSON).

In [ ]:
TOKENIZER_PATH = f'{WORK_DIR}/output/tokenizer/tokenizer_v2.json'
assert os.path.exists(TOKENIZER_PATH), (
    f'tokenizer_v2.json not found at {TOKENIZER_PATH}. '
    'Make sure output/tokenizer/tokenizer_v2.json is included in fant_code.zip.'
)

tok = FANT2Tokenizer.load(TOKENIZER_PATH)
sample = '<|im_start|>user\nWhat is 4+3?<|im_end|>\n<|im_start|>assistant\n<|answer|>\n7\n<|/answer|><|im_end|>'
ids = tok.encode(sample)
print(f'vocab={cfg.vocab_size}  sample len {len(sample)}B -> {len(ids)} tokens')
print('  decoded (first 120 chars):', tok.decode(ids)[:120])

## 6. Data pipeline

Streams the 6-source distillation mix (FineWeb, Opus-4.6, Kimi K2.5, NuminaMath, FineTome, Superior Reasoning) through the new `<answer>`-wrapped formatter. Datasets are cached to Drive so session restarts are cheap.

In [ ]:
from datasets import load_dataset
import random
import torch

# Mix v4 scale-aware (2026-04-19).
#   * For 20m/50m: CHAT-HEAVY (Sonnet 22%, Cascade-2 chat/IF, FineTome, Daring-Anteater)
#     — goal is fluent short-form chat like Qwen-2B, not reasoning benchmarks
#   * For 150m/350m/742m/1b: NVIDIA-heavy (MIX v3 preserved for reasoning-focused runs)
# All sources CC-BY-4.0 or permissive; decontaminated by 13-gram filter in cell 14.
#
# Columns:  (hf_id,  config,  split,  text_key,  format,  weight,  input_key)
MIX_V4_CHAT = [
    ('Roman1111111/claude-sonnet-4.6-120000x',            None,                    'train',  'messages',      DatasetFormat.MESSAGES,               0.22, None),
    ('crownelius/Opus-4.6-Reasoning-3300x',               None,                    'train',  'problem',       DatasetFormat.PROBLEM_THINK_SOLUTION, 0.14, None),
    ('ianncity/KIMI-K2.5-1000000x',                       'General-Distillation',  'train',  'messages',      DatasetFormat.MESSAGES,               0.12, None),
    ('nvidia/Nemotron-Cascade-2-SFT-Data',                'chat',                  'train',  'messages',      DatasetFormat.MESSAGES,               0.10, None),
    ('nvidia/Nemotron-Cascade-2-SFT-Data',                'instruction_following', 'train',  'messages',      DatasetFormat.MESSAGES,               0.06, None),
    ('mlabonne/FineTome-100k',                            None,                    'train',  'conversations', DatasetFormat.CONVERSATIONS,          0.08, None),
    ('nvidia/Daring-Anteater',                            None,                    'train',  'conversations', DatasetFormat.CONVERSATIONS,          0.05, None),
    ('HuggingFaceFW/fineweb-edu',                         'default',               'train',  'text',          DatasetFormat.FLAT_TEXT,              0.10, None),
    ('nvidia/OpenMathReasoning',                          None,                    'cot',    'problem',       DatasetFormat.PROBLEM_SOLUTION,       0.05, None),
    ('nvidia/OpenMathInstruct-2',                         None,                    'train',  'problem',       DatasetFormat.PROBLEM_SOLUTION,       0.04, None),
    ('nvidia/OpenCodeReasoning-2',                        None,                    'python', 'question',      DatasetFormat.PROBLEM_SOLUTION,       0.02, None),
    ('nvidia/Nemotron-Cascade-2-SFT-Data',                'math',                  'train',  'messages',      DatasetFormat.MESSAGES,               0.02, None),
]
# Total 1.00 — Sonnet 4.6 at 22% is the dominant quality signal.

MIX_V3_NVIDIA = [
    ('nvidia/OpenMathReasoning',                          None,                    'cot',            'problem',       DatasetFormat.PROBLEM_SOLUTION,       0.18, None),
    ('nvidia/OpenCodeReasoning-2',                        None,                    'python',         'question',      DatasetFormat.PROBLEM_SOLUTION,       0.08, None),
    ('nvidia/OpenMathInstruct-2',                         None,                    'train',          'problem',       DatasetFormat.PROBLEM_SOLUTION,       0.08, None),
    ('nvidia/Nemotron-Cascade-2-SFT-Data',                'math',                  'train',          'messages',      DatasetFormat.MESSAGES,               0.08, None),
    ('nvidia/Nemotron-Cascade-2-SFT-Data',                'chat',                  'train',          'messages',      DatasetFormat.MESSAGES,               0.06, None),
    ('nvidia/Nemotron-Cascade-2-SFT-Data',                'instruction_following', 'train',          'messages',      DatasetFormat.MESSAGES,               0.04, None),
    ('nvidia/Nemotron-Cascade-2-SFT-Data',                'science',               'train',          'messages',      DatasetFormat.MESSAGES,               0.04, None),
    ('nvidia/Daring-Anteater',                            None,                    'train',          'conversations', DatasetFormat.CONVERSATIONS,          0.04, None),
    ('HuggingFaceFW/fineweb-edu',                         'default',               'train',          'text',          DatasetFormat.FLAT_TEXT,              0.20, None),
    ('Roman1111111/claude-sonnet-4.6-120000x',            None,                    'train',          'messages',      DatasetFormat.MESSAGES,               0.12, None),
    ('crownelius/Opus-4.6-Reasoning-3300x',               None,                    'train',          'problem',       DatasetFormat.PROBLEM_THINK_SOLUTION, 0.08, None),
]

# Pick per-scale
if TARGET_SCALE in ('20m', '50m'):
    MIX = MIX_V4_CHAT
    print('Using MIX_V4_CHAT (Sonnet-heavy, chat-focused; for small-scale chat training)')
else:
    MIX = MIX_V3_NVIDIA
    print('Using MIX_V3_NVIDIA (NVIDIA reasoning-heavy; for larger-scale pretraining)')

def build_iterators(seed=0):
    iters = []
    weights = []
    for hf_id, cfg_, split, text_key, fmt, w, input_key in MIX:
        try:
            ds = load_dataset(hf_id, cfg_, split=split, streaming=True)
            ds = ds.shuffle(buffer_size=10_000, seed=seed)
        except Exception as e:
            print(f'  [skip] {hf_id}: {e}')
            continue
        kwargs = {'fmt': fmt, 'text_key': text_key}
        if fmt == DatasetFormat.INPUT_OUTPUT:
            kwargs['output_key'] = text_key
            if input_key is not None:
                kwargs['input_key'] = input_key
        iters.append((iter(ds), kwargs))
        weights.append(w)
    # Renormalise in case some skipped
    s = sum(weights)
    weights = [w / s for w in weights]
    return iters, weights

# Pad-token id — stored once for the loss masking below
_PAD_ID = tok._tok.token_to_id('<|pad|>')
if _PAD_ID is None: _PAD_ID = 0  # fallback (won't happen with our retrained tokenizer)

def sample_batch(iters, weights, batch_size, seq_len, tok, rng):
    """Return (input_ids, targets) both [B, T] int64.

    Target positions corresponding to padding are set to -100 so the CE loss
    ignores them (FANT3Model passes ignore_index=-100). Without this, the
    model trains to predict pad-token IDs at document tails and produces
    <|pad|><|pad|>... generations at inference time.
    """
    chunks = []
    attempts = 0
    while len(chunks) < batch_size and attempts < batch_size * 40:
        attempts += 1
        src_idx = rng.choices(range(len(iters)), weights=weights, k=1)[0]
        it, kwargs = iters[src_idx]
        try:
            ex = next(it)
        except StopIteration:
            continue
        try:
            text = extract_text(ex, **kwargs)
        except Exception:
            continue
        if not text or len(text) < 16:
            continue
        ids = tok.encode(text)
        if len(ids) < 8:
            continue
        need = seq_len + 1
        if len(ids) >= need:
            start = rng.randint(0, len(ids) - need)
            ids = ids[start : start + need]
        else:
            ids = ids + [_PAD_ID] * (need - len(ids))
        chunks.append(ids)
    if len(chunks) < batch_size:
        raise RuntimeError('data pipeline drained — check dataset IDs / HF cache')
    arr = torch.tensor(chunks, dtype=torch.long)
    input_ids = arr[:, :-1].contiguous()
    targets   = arr[:, 1:].contiguous().clone()
    # Mask pad positions out of the loss
    targets[targets == _PAD_ID] = -100
    return input_ids, targets

print('data pipeline ready')

## 7. Decontamination check

Runs `scripts/decontaminate.py` to flag any training docs that contain a 13-gram matching any test-set question of **GSM8K + MATH-500 + MMLU**. Local testing on the 6-source mix (n=1000 per source) shows NuminaMath-CoT 0.40%, Kimi/FineTome/Superior ~0.10%, FineWeb/Opus 0%. Install the filter into the data pipeline so contaminated docs never reach the model.


In [ ]:
# [FANT DECONTAMINATION CELL v1]
# Build (or load from disk) the benchmark n-gram signature cache. First run
# downloads the benchmark test sets (~50 MB total) and caches them to
# output/decontamination/ngram_hashes.json — subsequent runs are instant.
from scripts.decontaminate import build_hash_cache, is_contaminated, _load_global
_ = build_hash_cache(rebuild=False)
_ = _load_global()   # warms the global set

# Wrap the sample_batch() generator with a decontamination filter: any text that
# contains a 13-gram hash-matching any benchmark test question is dropped.
_original_sample_batch = sample_batch

def sample_batch(iters, weights, batch_size, seq_len, tok, rng):
    chunks = []
    attempts = 0
    rejected = 0
    while len(chunks) < batch_size and attempts < batch_size * 80:
        attempts += 1
        src_idx = rng.choices(range(len(iters)), weights=weights, k=1)[0]
        it, kwargs = iters[src_idx]
        try:
            ex = next(it)
        except StopIteration:
            continue
        try:
            text = extract_text(ex, **kwargs)
        except Exception:
            continue
        if not text or len(text) < 16:
            continue
        if is_contaminated(text):
            rejected += 1
            continue
        ids = tok.encode(text)
        if len(ids) < 8:
            continue
        need = seq_len + 1
        if len(ids) >= need:
            start = rng.randint(0, len(ids) - need)
            ids = ids[start : start + need]
        else:
            ids = ids + [_PAD_ID] * (need - len(ids))
        chunks.append(ids)
    if len(chunks) < batch_size:
        raise RuntimeError('data pipeline drained — check dataset IDs / HF cache')
    arr = torch.tensor(chunks, dtype=torch.long)
    input_ids = arr[:, :-1].contiguous()
    targets   = arr[:, 1:].contiguous().clone()
    targets[targets == _PAD_ID] = -100  # mask pad positions in loss
    return input_ids, targets

print(f'Decontamination filter active. Signatures loaded for GSM8K + MATH-500 + MMLU.')


## 8. Build model + optimizer

bf16 weights, 8-bit AdamW (bitsandbytes), gradient checkpointing where available.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# Enable gradient checkpointing automatically for large scales.
# At 742m+/T>=256 without gc, activations blow up past 90 GB.
cfg.use_gradient_checkpointing = TARGET_SCALE in ('742m', '1b')
print(f'use_gradient_checkpointing: {cfg.use_gradient_checkpointing}')

model = FANT3Model(cfg).to(device=device, dtype=dtype)
n_params = sum(p.numel() for p in model.parameters())
print(f'Built model: {n_params/1e6:.2f}M stored params on {device} {dtype}')

# NOTE: FANT3Model does not currently expose a gradient_checkpointing hook.
# If you hit OOM at larger scales, reduce BATCH_SIZE in the training cell
# before anything else. Adding torch.utils.checkpoint wrapping to MoEBlock/MoR
# is a future patch (see project_scale_ladder_warmup_2026_04_19.md).

# 8-bit AdamW — much lower optimizer-state VRAM cost than f32 AdamW
try:
    opt = bnb.optim.AdamW8bit(model.parameters(), lr=3e-4, betas=(0.9, 0.95),
                              weight_decay=0.1, eps=1e-8)
    OPT_KIND = 'bnb.AdamW8bit'
except Exception as e:
    print(f'bnb 8-bit failed ({e}), falling back to torch AdamW')
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1)
    OPT_KIND = 'torch.AdamW'
print(f'Optimizer: {OPT_KIND}')

## 9. Resume from the latest checkpoint (if any)

Checkpoints saved to Drive at `CKPT_DIR/<scale>/step_NNNN.pt` so session restarts don't lose progress.

In [ ]:
import glob, re

scale_ckpt_dir = os.path.join(CKPT_DIR, TARGET_SCALE)
os.makedirs(scale_ckpt_dir, exist_ok=True)

ckpts = sorted(
    glob.glob(os.path.join(scale_ckpt_dir, 'step_*.pt')),
    key=lambda p: int(re.search(r'step_(\d+)', p).group(1))
)

start_step = 0
losses_log = []
chirality_log = []

if ckpts:
    latest = ckpts[-1]
    ck = torch.load(latest, map_location=device)
    model.load_state_dict(ck['model'], strict=False)
    if 'opt' in ck:
        try:
            opt.load_state_dict(ck['opt'])
        except Exception as e:
            print(f'  opt restore skipped: {e}')
    start_step    = ck.get('step', 0)
    losses_log    = ck.get('losses', [])
    chirality_log = ck.get('chirality', [])
    print(f'Resumed from {latest} at step {start_step}, {len(losses_log)} log entries')
else:
    print('No checkpoint found — starting fresh')

## 10. Training loop

Defaults: **2500 steps** total, checkpoint every 250, log every 25. Change `TOTAL_STEPS` for longer runs.

In [ ]:
import os, time, math
# PYTORCH_CUDA_ALLOC_CONF is now set in cell 2 (before torch is imported).
# Setting it here is too late — CUDA is already initialized.

# ── NeMo / Nemotron-style recipe (2026-04-19) ───────────────────────────
# Based on Nemotron-4 and Llama-3 pretraining recipes adapted for Colab A100:
#   * Longer warmup (500 steps) — matches NVIDIA 2-5% of total, extra-safe for MoE
#   * Higher peak LR (2e-4) — halfway between NeMo's 3e-4 and our previous 1.5e-4
#   * Gradient accumulation = 4 -> effective batch 8 (closer to their 4M/step intent)
#   * AdamW betas (0.9, 0.95), weight_decay 0.1, eps 1e-8 — all NVIDIA defaults
# Scale-aware recipe (updated 2026-04-19 for 742m).  NVIDIA-style defaults;
# AdamW betas (0.9, 0.95), weight_decay 0.1, eps 1e-8 set in cell 16.
if TARGET_SCALE == '20m':
    # 20M chat-optimized, 12h budget on A100 96GB.
    # ~23.5M stored. Dense-ish (small MoE), no gc needed.
    BATCH_SIZE        = 16
    GRAD_ACCUM_STEPS  = 2       # effective batch = 32
    SEQ_LEN           = 1024
    TOTAL_STEPS       = 70000   # 12h budget at ~0.6 s/step
    WARMUP_STEPS      = 7000    # 10% of total
    peak_lr_setting   = 5.0e-4  # small models tolerate higher LR
elif TARGET_SCALE == '50m':
    # 50M chat-optimized, 12h budget on A100 96GB.
    # ~51M stored. Heavy over-training (~40x Chinchilla) to approach Qwen-2B chat quality.
    BATCH_SIZE        = 16
    GRAD_ACCUM_STEPS  = 2       # effective batch = 32
    SEQ_LEN           = 1024
    TOTAL_STEPS       = 60000   # 12h budget at ~0.7 s/step
    WARMUP_STEPS      = 6000    # 10% of total
    peak_lr_setting   = 4.0e-4
elif TARGET_SCALE == '150m':
    BATCH_SIZE        = 2
    GRAD_ACCUM_STEPS  = 4
    SEQ_LEN           = 512
    TOTAL_STEPS       = 2500
    WARMUP_STEPS      = 500
    peak_lr_setting   = 2.0e-4
elif TARGET_SCALE == '350m':
    BATCH_SIZE        = 2
    GRAD_ACCUM_STEPS  = 4
    SEQ_LEN           = 512
    TOTAL_STEPS       = 5000
    WARMUP_STEPS      = 750
    peak_lr_setting   = 1.8e-4
elif TARGET_SCALE == '742m':
    # Tier C (revised 2026-04-19 after Tier D hit 72 GB peak + gc fragmentation).
    # My earlier 40-50 GB VRAM estimate was wrong by ~2x: MatryoshkaMoE gathers
    # large W_up slices inside gc recompute, and the reserved-but-unallocated
    # pool grew to 48 GB by step 250.  Dropping B=2 -> B=1 (and bumping accum
    # 4 -> 8 to preserve effective batch) halves activation memory.
    BATCH_SIZE        = 1
    GRAD_ACCUM_STEPS  = 8       # effective batch = 8 (same as before)
    SEQ_LEN           = 1024
    TOTAL_STEPS       = 10000
    WARMUP_STEPS      = 1500
    peak_lr_setting   = 1.5e-4
else:  # '1b' — same memory-pressure lesson as 742m: B=1 safer at T=1024 with gc.
    BATCH_SIZE        = 1
    GRAD_ACCUM_STEPS  = 8       # effective batch = 8
    SEQ_LEN           = 1024
    TOTAL_STEPS       = 12000
    WARMUP_STEPS      = 1800
    peak_lr_setting   = 1.2e-4

LOG_EVERY           = 25
CKPT_EVERY          = 250
STORE_EVERY         = 50
GRAD_CLIP           = 1.0
print(f'Recipe: B={BATCH_SIZE} accum={GRAD_ACCUM_STEPS} seq={SEQ_LEN} '
      f'steps={TOTAL_STEPS} warmup={WARMUP_STEPS} peak_lr={peak_lr_setting:.1e}')

iters, weights = build_iterators(seed=42 + start_step)
rng = random.Random(123 + start_step)

def lr_at(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    # cosine down to 10% of peak over the remainder
    prog = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(prog, 1.0)))

model.train()
t0 = time.time()
training_diverged = False

for step in range(start_step, TOTAL_STEPS):
    # LR schedule
    lr_mul = lr_at(step)
    for g in opt.param_groups:
        g['lr'] = peak_lr_setting * lr_mul

    ids, targets = sample_batch(iters, weights, BATCH_SIZE, SEQ_LEN, tok, rng)
    ids = ids.to(device); targets = targets.to(device)

    out = model(ids, targets=targets, store_to_memory=(step % STORE_EVERY == 0))
    loss = out['loss']

    if not torch.isfinite(loss):
        print(f'[step {step}] non-finite loss — aborting (will NOT save final.pt)')
        training_diverged = True
        break

    # Gradient accumulation: scale loss before backward so the averaged gradient
    # over GRAD_ACCUM_STEPS micro-steps equals what a single larger batch would give.
    (loss / GRAD_ACCUM_STEPS).backward()
    if (step + 1) % GRAD_ACCUM_STEPS == 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP if step >= WARMUP_STEPS else 0.5)
        opt.step()
        opt.zero_grad(set_to_none=True)

    loss_val = loss.item()   # capture scalar before we free the graph
    losses_log.append((step, loss_val))

    if step % LOG_EVERY == 0:
        stats = model.memory.get_stats() if hasattr(model.memory, 'get_stats') else {}
        chirality_log.append((step, stats.get('chirality_balance', 0.5)))
        dt = time.time() - t0
        tps = BATCH_SIZE * SEQ_LEN * (step - start_step + 1) / max(dt, 1)
        vram = torch.cuda.max_memory_allocated() / 1e9 if device == 'cuda' else 0
        print(f'[{step:5d}/{TOTAL_STEPS}] ce {loss_val:6.3f}  '
              f"α {stats.get('alpha_fill','-'):>5} β {stats.get('beta_fill','-'):>5} "
              f"chir {stats.get('chirality_balance', 0.5):.3f}  "
              f'lr {g["lr"]:.2e}  tok/s {tps:7.0f}  vram {vram:.2f}GB', flush=True)

    if step > 0 and step % CKPT_EVERY == 0:
        path = os.path.join(scale_ckpt_dir, f'step_{step:05d}.pt')
        from dataclasses import asdict as _asdict
        torch.save({
            'step': step,
            'model': model.state_dict(),
            'opt': opt.state_dict(),
            'losses': losses_log,
            'chirality': chirality_log,
            'cfg_scale': TARGET_SCALE,
            'cfg_dict': _asdict(cfg),
        }, path)
        print(f'  saved {path}')
        torch.cuda.empty_cache()   # release reserved-but-unallocated segments after save

    # End-of-iteration cleanup: free the backward-graph references held by the
    # out dict so next iteration allocates afresh (crucial on A100 near capacity).
    del out, loss, ids, targets

# Final save — skip if training diverged (corrupt weights would overwrite good ckpts)
final_path = os.path.join(scale_ckpt_dir, 'final.pt')
if training_diverged:
    print(f'Done (DIVERGED at step {step}). Wall time {(time.time()-t0)/60:.1f} min.')
    print(f'NOT saving final.pt — use the latest step_*.pt checkpoint instead (~step {max(0,(step//CKPT_EVERY)*CKPT_EVERY)})')
else:
    # Save full cfg dict so eval/resume can rebuild the model without knowing the scale preset
    from dataclasses import asdict
    torch.save({'step': step, 'model': model.state_dict(), 'losses': losses_log,
                'chirality': chirality_log, 'cfg_scale': TARGET_SCALE,
                'cfg_dict': asdict(cfg)}, final_path)
    print(f'Done. Wall time {(time.time()-t0)/60:.1f} min. Final {final_path}')

## 11. Loss + chirality plots

In [ ]:
import matplotlib.pyplot as plt

if losses_log:
    steps_l, vals_l = zip(*losses_log)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(steps_l, vals_l, lw=0.7)
    axes[0].set_xlabel('step'); axes[0].set_ylabel('CE loss')
    axes[0].set_title(f'FANT 3 {TARGET_SCALE} cross-entropy')
    axes[0].axhline(4.0, ls='--', alpha=0.4, label='CE=4 coherence threshold')
    axes[0].legend()

    if chirality_log:
        steps_c, vals_c = zip(*chirality_log)
        axes[1].plot(steps_c, vals_c, lw=1.0, color='tab:orange')
        axes[1].set_xlabel('step'); axes[1].set_ylabel('α/β chirality balance')
        axes[1].set_title('Spinor Apollonian chirality split')
        axes[1].axhline(0.5, ls='--', alpha=0.4, color='k')
        axes[1].set_ylim(0, 1)
    plt.tight_layout(); plt.show()
else:
    print('No logs yet')

## 12. Quick quality probe

Run a handful of arithmetic prompts through the trained model and see whether it emits a valid `<|answer|>...<|/answer|>` structure. Compare to the 6% at-chance result from the pre-fix FANT 2 step_3000 run.

In [ ]:
from fant2.tokenizer.chat_template import apply_chat_template

probes = [
    'What is 4 + 3?',
    'What is 9 * 7?',
    'Solve for x: 2x + 5 = 17.',
    'The sum of 12 and 8 is what?',
    'If a pizza has 8 slices and I eat 3, how many are left?',
]

model.eval()
for q in probes:
    prompt = apply_chat_template(
        [{'role': 'user', 'content': q}],
        add_generation_prompt=True, add_bos=True,
    )
    ids = torch.tensor([tok.encode(prompt)], device=device)
    with torch.no_grad():
        for _ in range(64):  # greedy 64 tokens
            out = model(ids)
            nxt = out['logits'][:, -1].argmax(dim=-1, keepdim=True)
            ids = torch.cat([ids, nxt], dim=1)
            tid = nxt.item()
            stop_ids = {tok._tok.token_to_id(t) for t in ('<|im_end|>', '<|eos|>', '<|pad|>')}
            stop_ids.discard(None)
            if tid in stop_ids or ids.shape[1] > 256:
                break
    text = tok.decode(ids[0].tolist())
    completion = text[len(prompt):]
    print('Q:', q)
    print('A:', completion[:200].replace('\n', ' | '))
    print()
model.train()

## 13. Benchmark eval

After training, evaluate on the three benchmarks our training mix has been decontaminated against. Run on the smallest eval size that gives a meaningful signal; scale up if the accuracy looks promising.


In [ ]:
# [FANT BENCHMARK EVAL CELL v1]
# Run GSM8K + MMLU eval. Expect sub-chance from a 50m/150m run of 2500 steps —
# the point is to get a DIRECTION and a baseline to improve on with scale.
import subprocess, json, os

# Save a checkpoint path the eval script understands
eval_ckpt = os.path.join(scale_ckpt_dir, 'final.pt')
if not os.path.exists(eval_ckpt):
    # pick the latest step_*.pt
    ckpts = sorted([p for p in os.listdir(scale_ckpt_dir) if p.startswith('step_')],
                   key=lambda p: int(p.split('_')[1].split('.')[0]))
    eval_ckpt = os.path.join(scale_ckpt_dir, ckpts[-1]) if ckpts else None

if eval_ckpt is None:
    print('No checkpoint yet — run training first')
else:
    print(f'Evaluating {eval_ckpt}')
    for bench, n in [('gsm8k', 50), ('mmlu', 200)]:
        print(f'\n--- {bench} ({n} problems) ---')
        result = !python {WORK_DIR}/scripts/eval_benchmarks.py \
            --ckpt {eval_ckpt} \
            --tokenizer {TOKENIZER_PATH} \
            --benchmark {bench} \
            --n {n}
        print('\n'.join(result[-10:]))


## 14. Scale up

Once 50m shows clean loss descent (CE trending below ~6 with recognisable chirality split staying non-degenerate), change `TARGET_SCALE` at the top of cell 4 and re-run from there.

**Scaling guidance on Colab A100 40 GB:**

| scale | batch | seq | steps | approx walltime | VRAM |
|---|---|---|---|---|---|
| `50m` | 8 | 512 | 2500 | ~15 min | ~4 GB |
| `150m` | 8 | 1024 | 5000 | ~45 min | ~8 GB |
| `350m` | 4 | 1024 | 10000 | ~2 h | ~15 GB |
| `742m` | 4 | 1024 | 20000 | ~5 h | ~25 GB (needs A100 40) |
| `1b`   | 2 | 1024 | 50000 | ~12 h | ~35 GB (needs A100 80) |

On an H100, divide those wall times by ~3. On a T4 (free tier), multiply by ~3 and only the first two scales are practical.